In [ ]:
import os
from dotenv import set_key, load_dotenv


load_dotenv(".env")

repo_code = os.getenv("PCLOUD_CODE")
pcloud_username = os.getenv("PCLOUD_USERNAME")
pcloud_password = os.getenv("PCLOUD_PASSWORD")

In [1]:
from lcdb.db import PCloudRepository
repo = PCloudRepository(repo_code=repo_code)
repo.authenticate(username=pcloud_username, password=pcloud_password, authexpire=86400*2)

dotenv_path = ".env"  
set_key(dotenv_path, "PCLOUD_TOKEN", repo.token)


NameError: name 'repo_code' is not defined

In [ ]:
repo_token = os.getenv("PCLOUD_TOKEN")

In [ ]:
repo = PCloudRepository(repo_code=repo_code, token=repo_token)

repo code: kZeWywZRr6lScWSloHlzwk6Uxq3GyRtuBaX


In [ ]:
workflow_mapping = {
  "libsvm": "lcdb.workflow.sklearn.LibSVMWorkflow",
  "randomforest": "lcdb.workflow.sklearn.RandomForestWorkflow",
  "knn": "lcdb.workflow.sklearn.KNNWorkflow",
  "xgboost": "lcdb.workflow.xgboost.XGBoostWorkflow",
  "treesensemble": "lcdb.workflow.sklearn.TreesEnsembleWorkflow",
  "liblinear": "lcdb.workflow.sklearn.LibLinearWorkflow"
}

workflow = 'liblinear'
workflow_class = workflow_mapping[workflow]
memory = 30

def get_folder_id(workflow, memory):
  return repo._get_folder_id(f"data/{workflow}/data_probing-{memory}")

folder_id = get_folder_id(workflow_class, memory)

In [ ]:
import requests

response = requests.get(
    f"https://eapi.pcloud.com/listfolder?code={repo.repo_code}&auth={repo.token}&folderid=14175375152"
).json()
print(response)

{'result': 0, 'metadata': {'name': 'lcdb', 'created': 'Thu, 05 Dec 2024 09:11:31 +0000', 'ismine': True, 'thumb': False, 'modified': 'Thu, 05 Dec 2024 09:11:39 +0000', 'comments': 0, 'id': 'd14175375152', 'isshared': False, 'icon': 'folder', 'isfolder': True, 'parentfolderid': 14175372679, 'folderid': 14175375152, 'contents': [{'name': 'data', 'created': 'Thu, 05 Dec 2024 09:11:39 +0000', 'ismine': True, 'thumb': False, 'modified': 'Mon, 03 Mar 2025 13:34:47 +0000', 'comments': 0, 'id': 'd14175377067', 'isshared': False, 'icon': 'folder', 'isfolder': True, 'parentfolderid': 14175375152, 'folderid': 14175377067}]}}


In [ ]:
repo.content

{'result': 0,
 'metadata': {'name': 'data_probing-30',
  'created': 'Thu, 20 Mar 2025 09:09:30 +0000',
  'ismine': True,
  'thumb': False,
  'modified': 'Thu, 20 Mar 2025 09:14:49 +0000',
  'comments': 0,
  'id': 'd15914226140',
  'isshared': False,
  'icon': 'folder',
  'isfolder': True,
  'parentfolderid': 14175848526,
  'folderid': 15914226140,
  'contents': [{'name': '1049',
    'created': 'Thu, 20 Mar 2025 09:10:00 +0000',
    'ismine': True,
    'thumb': False,
    'modified': 'Thu, 20 Mar 2025 09:10:00 +0000',
    'comments': 0,
    'id': 'd15914236257',
    'isshared': False,
    'icon': 'folder',
    'isfolder': True,
    'parentfolderid': 15914226140,
    'folderid': 15914236257},
   {'name': '1067',
    'created': 'Thu, 20 Mar 2025 09:10:04 +0000',
    'ismine': True,
    'thumb': False,
    'modified': 'Thu, 20 Mar 2025 09:10:04 +0000',
    'comments': 0,
    'id': 'd15914237487',
    'isshared': False,
    'icon': 'folder',
    'isfolder': True,
    'parentfolderid': 15914

In [ ]:
import requests
import csv

def get_subfolder_names(auth_token, repo_code, folder_id):
    url = "https://eapi.pcloud.com/listfolder"
    params = {
        "code": repo_code,
        "auth": auth_token,
        "folderid": folder_id
    }

    response = requests.get(url, params=params).json()

    if response.get("result") == 0:
        try:
            # Extract and sort folder names numerically
            folders = sorted(
                (item["name"] for item in response["metadata"].get("contents", []) if item.get("isfolder")),
                key=int  # Convert to int for correct numerical sorting
            )
            return folders
        except ValueError:
            print("Warning: Some folder names are not purely numeric. Falling back to default sorting.")
            return sorted(
                (item["name"] for item in response["metadata"].get("contents", []) if item.get("isfolder"))
            )
    else:
        print(f"Error: {response.get('error', 'Unknown error')}")
        return []

def save_to_csv(folders, filename="folders.csv"):
    with open(filename, mode="w", newline="") as file:
        writer = csv.writer(file)
        # writer.writerow(["Folder Name"])  # Add header row
        for folder in folders:
            writer.writerow([folder])
    print(f"Saved {len(folders)} folders to {filename}")

# Example usage
auth_token = repo.token
repo_code = repo.repo_code

folders = get_subfolder_names(auth_token, repo_code, folder_id)

if folders:
    save_to_csv(folders, filename="liblinear-30-datasets.csv")
    print(f"Subfolders: {folders}")
